# FinReasoningAI Colab
End-to-end FinCoT QLoRA training for `Qwen/Qwen2.5-14B-Instruct`, with a standalone PEFT adapter saved for vLLM LoRA serving.

## Step 0a: GPU Check
Expected runtime: under 1 minute. VRAM guidance: training is designed for an A100 with at least 35 GB free VRAM before model load.

In [1]:
import subprocess

def gpu_summary():
    try:
        out = subprocess.check_output([
            "nvidia-smi",
            "--query-gpu=name,memory.total,memory.free",
            "--format=csv,noheader,nounits",
        ], text=True)
        print(out.strip())
        first = out.strip().splitlines()[0].split(",")
        free_gb = float(first[2].strip()) / 1024.0
        if free_gb < 35:
            print(f"Warning: only {free_gb:.1f} GB free VRAM detected; training may OOM.")
    except Exception as exc:
        print(f"Unable to query GPU details: {exc}")

gpu_summary()

NVIDIA A100-SXM4-40GB, 40960, 40442


## Step 0b: Mount Google Drive and Infer Workspace
Expected runtime: 1-2 minutes. This cell mounts Drive and auto-detects the notebook workspace instead of hardcoding paths.

In [2]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_BASE = "/content/drive/MyDrive/FinReasoningAI"
PROJECT_DIR = f"{DRIVE_BASE}/FinReasoningAI"

print("DRIVE_BASE:", DRIVE_BASE)
print("PROJECT_DIR:", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_BASE: /content/drive/MyDrive/FinReasoningAI
PROJECT_DIR: /content/drive/MyDrive/FinReasoningAI/FinReasoningAI


## Step 0c: Install Missing Dependencies
Expected runtime: 3-8 minutes on a fresh runtime. PyTorch is intentionally not installed here because Colab already provides the GPU build.

In [3]:
import importlib
import pkg_resources
import subprocess
import sys

REQUIRED = {
    "transformers":  "4.41.0",
    "datasets":      "2.19.0",
    "accelerate":    "0.30.0",
    "peft":          "0.10.0",
    "trl":           "0.8.6",
    "bitsandbytes":  "0.44.0",
    "evaluate":      "0.4.1",
    "rouge-score":   "0.1.2",
    "scikit-learn":  "1.4.0",
    "pandas":        "2.2.0",
    "pydantic":      "2.7.0",
    "jsonlines":     "4.0.0",
    "bitsandbytes": "0.46.1"
}

missing = []
for package, version in REQUIRED.items():
    try:
        installed = pkg_resources.get_distribution(package).version
        if installed != version:
            missing.append(f"{package}=={version}")
    except Exception:
        missing.append(f"{package}=={version}")

if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    print("Restart the runtime before training if packages were freshly installed.")
else:
    print("All required packages already match the requested versions.")

All required packages already match the requested versions.


/tmp/ipykernel_6296/3137331769.py:2: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


## Step 0d: Clone or Pull the Repository
Expected runtime: under 2 minutes. This cell always refreshes the repo in the current Colab workspace before imports.

In [15]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/juankim834/FinReasoningAI.git"
DRIVE_BASE = "/content/drive/MyDrive/FinReasoningAI"
PROJECT_DIR = Path(DRIVE_BASE) / "FinReasoningAI"

if (PROJECT_DIR / ".git").exists():
    subprocess.check_call(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"])
else:
    raise FileNotFoundError(
        f"Expected repo at {PROJECT_DIR}, but .git was not found. "
        "Please ensure the real repository lives in DRIVE_BASE/FinReasoningAI."
    )

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("PROJECT_DIR:", PROJECT_DIR)

PROJECT_DIR: /content/drive/MyDrive/FinReasoningAI/FinReasoningAI


## Step 1: Configuration
Expected runtime: immediate. Adjust the knobs here before running the data, training, and evaluation cells below.

In [5]:
# MODEL
MODEL_ID          = "Qwen/Qwen2.5-14B-Instruct"

# DATA
MAX_SAMPLES       = None            # None = full SFT split
INCLUDE_COT       = True
FINAL_SAMPLE_SIZE = 5000
TRAIN_SIZE        = 4500
TEST_SIZE         = 500
SEED              = 42
PROCESSED_DATA_DIR = "data/processed_fincot_sft"

# TRAINING
# A100 40 GB: BATCH_SIZE=2, GRAD_ACCUM=16 (effective batch = 32, same as BATCH_SIZE=4 + GRAD_ACCUM=8).
# A100 80 GB: BATCH_SIZE=4, GRAD_ACCUM=8 is safe.
NUM_EPOCHS        = 1
BATCH_SIZE        = 2
GRAD_ACCUM        = 16
GRAD_CKPT         = True   # trades compute for memory, ~30-40% memory reduction
LEARNING_RATE     = 2e-4
MAX_SEQ_LEN       = 2048
# Samples longer than MAX_TRAIN_LEN tokens are hard-dropped before training.
# With the ChatML assistant-marker fix, truncation to MAX_SEQ_LEN is safe for
# any sample, so set this high (16384) to only discard truly pathological entries.
# The original 4096 default caused ~36% of FinQA long-context samples to be dropped.
MAX_TRAIN_LEN     = 16384
USE_WANDB         = False

# LoRA
LORA_R            = 64
LORA_ALPHA        = 128
LORA_DROPOUT      = 0.05
LORA_TARGET_MODS  = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# OUTPUT
OUTPUT_DIR        = "outputs/sft_qlora"
ADAPTER_SAVE_DIR  = f"{OUTPUT_DIR}/final_adapter"
EVAL_MAX_SAMPLES  = 200
SKIP_DPO          = True



## Step 2a: Load the Base Model
Expected runtime: 5-10 minutes. VRAM usage after 4-bit load is typically around 10-15 GB before LoRA adapters and activations.

In [6]:
!pip uninstall -y bitsandbytes
!pip install -U bitsandbytes==0.46.1

Found existing installation: bitsandbytes 0.46.1
Uninstalling bitsandbytes-0.46.1:
  Successfully uninstalled bitsandbytes-0.46.1
  Using cached bitsandbytes-0.46.1-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.46.1-py3-none-manylinux_2_24_x86_64.whl (72.9 MB)


In [7]:
import os
os.environ["FINREASONING_USE_BF16"] = "1"

from src.model.load_model import get_default_bnb_config, get_preferred_torch_dtype, load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer(
    model_id=MODEL_ID,
    bnb_config=get_default_bnb_config(),
)

import gc
import torch
if torch.cuda.is_available():
    print(f"Allocated VRAM after base load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print(f"Preferred dtype: {get_preferred_torch_dtype()}")
print(f"bitsandbytes compute dtype: {get_default_bnb_config().bnb_4bit_compute_dtype}")

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Base model loader check passed. Training and eval cells will load models explicitly when needed.")



Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Allocated VRAM after base load: 10.84 GB


## Step 2b: Apply QLoRA
Expected runtime: under 2 minutes. This attaches trainable adapters while keeping the base model quantized for A100-friendly fine-tuning.

In [8]:
import os
os.environ["FINREASONING_USE_BF16"] = "1"

from src.model.apply_lora import apply_qlora
from src.model.load_model import get_default_bnb_config, load_model_and_tokenizer
from src.train.sft_train import build_lora_config

model, tokenizer = load_model_and_tokenizer(
    model_id=MODEL_ID,
    bnb_config=get_default_bnb_config(),
)

lora_config = build_lora_config(
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    lora_target_modules=LORA_TARGET_MODS,
)
model = apply_qlora(model, lora_config=lora_config, gradient_checkpointing=True)
model.print_trainable_parameters()



trainable params: 275,251,200 || all params: 15,045,284,864 || trainable%: 1.8294848019701808


## Step 3a: Load FinCoT Samples
Expected runtime: 1-5 minutes depending on whether the dataset is pulled from Hugging Face or read locally.

In [9]:
from src.data.fincot_loader import load_fincot_samples

samples, dataset_meta = load_fincot_samples(max_samples=MAX_SAMPLES)
print(dataset_meta)
assert len(samples) > 0
assert all(key in samples[0] for key in ["question", "answer", "reasoning"])



Dataset source: gbharti/finance-alpaca
Loaded 4902 FinCoT samples.
Task distribution: financial_qa=2924, numerical_reasoning=1765, structured_analysis=213


## Step 3b: Inspect One Sample per Task
Expected runtime: under 1 minute. This helps confirm the loader is producing the normalized schema we expect before formatting.

In [10]:
print(samples[0])



financial_qa
{'question': 'Describe the effects of climate change.', 'answer': "Climate change is having a profound effect on the environment and all life on Earth. It is causing higher temperatures across the globe, extreme weather changes such as flooding and drought, and an increase in sea levels. In addition to these physical effects, climate change also has an emotional impact. It causes anxiety and stress, due to the worry of the unknown and the potential destruction it could cause. Finally, it is disproportionately affecting low-income households, which don't have the resources to access clean energy and adapt to the changing environment.", 'chain_of_thought': '', 'context': None, 'task': 'financial_qa'}
numerical_reasoning
{'question': 'Create an example of outcomes from working in a team environment.', 'answer': 'Working in a team environment can lead to increased efficiency, improved decision-making, better problem-solving skills, higher productivity, and more creative soluti

## Step 3c: Build Prompt/Completion Datasets
Expected runtime: 2-5 minutes. The preprocessing path uses `tokenizer.apply_chat_template` so the chat format stays aligned with Qwen tokenizer updates.

In [11]:
import numpy, bitsandbytes, torch
print("numpy:", numpy.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("cuda:", torch.cuda.is_available())

numpy: 1.26.4
bitsandbytes: 0.46.1
cuda: True


In [18]:
from pathlib import Path

from src.data.preprocess import prepare_fincot_sft_dataset
from src.train.sft_train import load_datasets, _tokenize_old_trl_dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

if not Path(PROCESSED_DATA_DIR).exists():
    print(f"{PROCESSED_DATA_DIR} not found. Building the processed dataset first...")
    prepare_fincot_sft_dataset(
        tok,
        output_dir=PROCESSED_DATA_DIR,
        final_sample_size=FINAL_SAMPLE_SIZE,
        train_size=TRAIN_SIZE,
        test_size=TEST_SIZE,
        include_cot=INCLUDE_COT,
        seed=SEED,
    )

ds = load_datasets(PROCESSED_DATA_DIR)

# --- dataset quality check ---
from src.train.sft_train import _prepare_old_trl_dataset, _ASSISTANT_MARKER
raw_train = ds["train"]
n_total = len(raw_train)
n_no_chatml = sum(1 for ex in raw_train if _ASSISTANT_MARKER not in str(ex.get("prompt", "")))
print(f"Training samples          : {n_total}")
print(f"Samples lacking ChatML    : {n_no_chatml} ({100*n_no_chatml/n_total:.1f}%)  <- will be re-wrapped")

# Tokenize with the same settings used during sft_main to see how many get dropped.
tmp = _tokenize_old_trl_dataset(ds["train"], tok,
                                max_seq_length=MAX_SEQ_LEN,
                                max_train_length=MAX_TRAIN_LEN)
print(f"After filtering           : {len(tmp)} samples kept")
print(f"Dropped (>{MAX_TRAIN_LEN} tokens)  : {n_total - len(tmp)}")
print(tmp.column_names)
print({k: type(v).__name__ for k, v in tmp[0].items()})



Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Map:   0%|          | 0/4410 [00:00<?, ? examples/s]

['input_ids', 'attention_mask', 'special_tokens_mask']
{'input_ids': 'list', 'attention_mask': 'list', 'special_tokens_mask': 'list'}


In [17]:
from src.data.preprocess import prepare_fincot_sft_dataset, print_preparation_summary, format_sample_as_chat

dataset, dataset_summary = prepare_fincot_sft_dataset(
    tokenizer,
    output_dir=PROCESSED_DATA_DIR,
    final_sample_size=FINAL_SAMPLE_SIZE,
    train_size=TRAIN_SIZE,
    test_size=TEST_SIZE,
    include_cot=INCLUDE_COT,
    seed=SEED,
)
print_preparation_summary(dataset_summary, PROCESSED_DATA_DIR)
print({split: len(ds) for split, ds in dataset.items()})
example = format_sample_as_chat(samples[0], tokenizer, include_cot=INCLUDE_COT)
print(example["prompt"][:1500])
print("\n--- COMPLETION ---\n")
print(example["completion"])
assert example["prompt"]
assert example["completion"]



Saving the dataset (0/1 shards):   0%|          | 0/4410 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/244 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/248 [00:00<?, ? examples/s]

{'train': 4410, 'val': 244, 'test': 248}
<|im_start|>system
You are FinReasoningAI, a financial reasoning assistant. Think step by step before giving your final answer. For numerical questions, show your calculation chain. Your final answer must be on a line starting with 'Answer:'.

## Available Tools
[
  {
    "type": "function",
    "function": {
      "name": "calculate_financial_ratio",
      "description": "Compute a standard financial ratio from two numeric values.",
      "parameters": {
        "type": "object",
        "properties": {
          "numerator": {
            "type": "number",
            "description": "The numerator value"
          },
          "denominator": {
            "type": "number",
            "description": "The denominator value"
          },
          "ratio_name": {
            "type": "string",
            "description": "E.g. 'P/E ratio', 'gross margin'"
          }
        },
        "required": [
          "numerator",
          "denominator",


## Step 4: SFT Training (1 Epoch)
Expected runtime: several hours on an A100 depending on dataset size. Peak VRAM will usually sit in the 30-40 GB range with the default sequence length and effective batch size.

In [14]:
from src.train.sft_train import load_datasets, _prepare_old_trl_dataset
ds = load_datasets(PROCESSED_DATA_DIR)
tmp = _prepare_old_trl_dataset(ds["train"], tokenizer=tok)
print(tmp.column_names)
print(tmp[0].keys())



['text']
dict_keys(['text'])


In [13]:
import gc
import importlib
import os
import torch

os.environ["FINREASONING_USE_BF16"] = "1"

import src.model.load_model as load_model
import src.train.sft_train as sft_train

importlib.reload(load_model)
importlib.reload(sft_train)

from src.train.sft_train import main as sft_main, save_adapter_for_vllm

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print(f"Preferred dtype for this runtime: {load_model.get_preferred_torch_dtype()}")
print(f"bitsandbytes compute dtype: {load_model.get_default_bnb_config().bnb_4bit_compute_dtype}")

trainer = sft_main(
    model_id=MODEL_ID,
    output_dir=OUTPUT_DIR,
    data_dir=PROCESSED_DATA_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=GRAD_CKPT,
    learning_rate=LEARNING_RATE,
    max_seq_length=MAX_SEQ_LEN,
    max_train_length=MAX_TRAIN_LEN,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    lora_target_modules=LORA_TARGET_MODS,
    use_wandb=USE_WANDB,
)
save_adapter_for_vllm(trainer=trainer, output_dir=ADAPTER_SAVE_DIR)



Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Map:   0%|          | 0/4410 [00:00<?, ? examples/s]

Map:   0%|          | 0/244 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:487: UserWarning: You passed `remove_unused_columns=False` on a non-packed dataset. This might create some issues with the default collator and yield to errors. If you want to inspect dataset other columns (in this case ['text']), you can subclass `DataCollatorForLanguageModeling` in case you used the default collator and create your own data collator in order to inspect the unused dataset columns.
  warnings.warn(


Map:   0%|          | 0/4410 [00:00<?, ? examples/s]

Map:   0%|          | 0/244 [00:00<?, ? examples/s]

ValueError: Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`text` in this case) have excessive nesting (inputs type `list` where type `int` is expected).

## Step 5: Evaluation
Expected runtime: 10-30 minutes for 200 samples depending on decoding mode. This reuses the raw held-out test slice so answer/context fields are available for metrics.

In [ ]:
from pathlib import Path

import gc
import torch

# ── Eval-only restart: run Step 0b → 0c → 0d → Step 1 → this cell. ──────────
# Training (Step 4) is NOT required; adapter and processed data must already
# exist on Drive from a prior run.
from pathlib import Path
assert Path(ADAPTER_SAVE_DIR).exists(), (
    f"Adapter not found at {ADAPTER_SAVE_DIR!r}. "
    "Run Step 4 (training) first, or point ADAPTER_SAVE_DIR to an existing adapter."
)
assert Path(PROCESSED_DATA_DIR).exists(), (
    f"Processed dataset not found at {PROCESSED_DATA_DIR!r}. "
    "Run Step 3c (data prep) first, or point PROCESSED_DATA_DIR to an existing dataset."
)

from src.data.preprocess import load_eval_test_samples
from src.eval.evaluate import evaluate_model
from src.model.load_model import load_model_with_adapter

if "trainer" in globals():
    del trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

eval_model, eval_tokenizer = load_model_with_adapter(MODEL_ID, ADAPTER_SAVE_DIR)
test_samples = load_eval_test_samples(PROCESSED_DATA_DIR)
metrics = evaluate_model(
    eval_model,
    eval_tokenizer,
    test_samples,
    output_csv="outputs/eval_results.csv",
    max_samples=EVAL_MAX_SAMPLES,
)
print(metrics)



## Step 6a: Direct Inference Demo
Expected runtime: under 1 minute for a single prompt. This is the shortest path for a basic financial QA response.

In [ ]:
from src.inference.generate import build_prompt, generate_answer
from src.model.load_model import load_model_with_adapter

question = "What does a lower debt-to-equity ratio generally suggest about a company's balance sheet?"
if "eval_model" not in globals() or "eval_tokenizer" not in globals():
    eval_model, eval_tokenizer = load_model_with_adapter(MODEL_ID, ADAPTER_SAVE_DIR)
prompt = build_prompt(question=question, tokenizer=eval_tokenizer)
print(prompt[:1000])
print()
print(generate_answer(eval_model, eval_tokenizer, question=question, max_new_tokens=128, grounding_check=False))



## Step 6b: Chain-of-Thought Inference Demo
Expected runtime: under 1 minute. This uses the reasoning-oriented inference path and expects the answer to end with an `Answer:` line.

In [ ]:
cot_question = "If revenue grows from 120 to 150, what is the percentage growth?"
print(generate_answer(model, tokenizer, question=cot_question, use_cot=True, max_new_tokens=256, grounding_check=False))

## Step 6c: Tool-Augmented Inference Demo
Expected runtime: under 1 minute. This exercises the tool loop so we can confirm a ratio question either triggers a tool call or still returns a plain answer without crashing.

In [ ]:
from src.inference.generate import generate_with_tools
from src.model.load_model import load_model_with_adapter
from tools.financial_tools import FINANCIAL_TOOLS

if "eval_model" not in globals() or "eval_tokenizer" not in globals():
    eval_model, eval_tokenizer = load_model_with_adapter(MODEL_ID, ADAPTER_SAVE_DIR)

tool_demo = generate_with_tools(
    eval_model,
    eval_tokenizer,
    question="What is Apple's P/E ratio given net income of 97 and market cap of 2800?",
    tools=FINANCIAL_TOOLS,
    max_new_tokens=256,
)
print(tool_demo)



## Step 6d: Self-Consistency Demo
Expected runtime: a few minutes for 5 samples because each prompt is sampled multiple times. Use this to inspect agreement-based robustness after training.

In [ ]:
from src.inference.self_consistency import sample_with_self_consistency

prompt = build_prompt(question="What is the CAGR from 100 to 121 over 2 periods?", tokenizer=tokenizer, use_cot=True)
final_answer, confidence, raw_answers = sample_with_self_consistency(
    model, tokenizer, prompt, n=5, temperature=0.7, max_new_tokens=128
)
print("Final:", final_answer)
print("Confidence:", confidence)
print(raw_answers)

## Step 7: Optional DPO
Expected runtime: only relevant if you later choose to extend the pipeline beyond the 1-epoch SFT run. This stays disabled by default.

In [ ]:
if SKIP_DPO:
    print("Skipping DPO as configured.")
else:
    print("Add your optional DPO workflow here.")

## Verification Checklist
Expected runtime: under 1 minute after earlier cells complete. These assertions cover the minimum smoke tests requested in the rewrite spec.

In [ ]:
from pathlib import Path

assert len(samples) > 0
formatted = format_sample_as_chat(samples[0], tokenizer, include_cot=INCLUDE_COT)
assert formatted["prompt"] and formatted["completion"]
assert set(dataset.keys()) == {"train", "test"}
assert sum(len(ds) for ds in dataset.values()) == FINAL_SAMPLE_SIZE
assert Path(ADAPTER_SAVE_DIR).exists()
assert Path(ADAPTER_SAVE_DIR, "adapter_config.json").exists()
assert isinstance(tool_demo["tool_calls"], list)
print("Notebook smoke checks passed.")

